Week 3 (14/03/2025) - Building a document scanner with gradio

This implementation makes use of a GPU, defined through the gradio library, in order to run the scanner even outside of a local Python script.

In [1]:
import gradio as gr

# Start by defining a foo() function for the interface.
# This function takes two inputs in order to match how many inputs are specified in the interface, which will then store the returned value in the output.
def foo(name, intensity):
    return 'Hello, ' + name + '!' * int(intensity)

# Now, define the gradio interface by specifying the function to call, the inputs and the outputs.
demo = gr.Interface(
    fn=foo, 
    inputs=['text', 'slider'], 
    outputs=['text'])

# Lastly, launch the interface for execution through the launch() function.
demo.launch()
# N.B.: If the interface is run on the terminal, it can be closed by pressing Ctrl+C.

* Running on local URL:  http://127.0.0.1:7861

To create a public link, set `share=True` in `launch()`.


Created dataset file at: .gradio/flagged/dataset1.csv


Observe that it is possible to employ blocks in order to build more complex GUIs as well.

In [1]:
import gradio as gr
import numpy as np
import cv2

# Let src_points be the variable that defines the input corner and initialise it as an empty array that will be filled at runtime.
src_points = []

# Now, define the onSelect() function for taking the corners from the image to scan.
def onSelect(value, evt: gr.EventData):
    if len(src_points) < 4:
        src_points.append(evt._data['index']) # to get the point on which the event occurs
    return len(src_points)

# Start by defining the fixImg() function for the GUI.
def fixImg(img):
    dst_points = np.float32([[0, 0], [0, 800], [600, 800], [600, 0]]) # destination points for a 800x600 image
    src_float = np.float32(src_points) # convert src_points from a standard list to a numpy array to avoid errors
    H = cv2.getPerspectiveTransform(src_float, dst_points) # get the perspective matrix
    output_img = cv2.warpPerspective(img, H, (600, 800)) # apply the transformation to obtain a 800x600 image
    return output_img

# Consider creating a document scanner using this blocks approach.
with gr.Blocks() as demo:
    gr.Markdown('Document Scanner') # to add text in the interface
    coord_n = gr.Textbox(label='Number of click coordinates', value=0) # textbox to keep track of the clicked corners (the value is set to 0 by default)
    # Now, start defining rows for keeping inputs and outputs through the Row() gradio function (note that the row space will be equally divided among its elements).
    # The fisrt row will contain the input and output image.
    with gr.Row():
        inp = gr.Image(label='Input') # upload the image to be fixed
        inp.select(fn=onSelect, inputs=inp, outputs=coord_n) # event for getting the corners of the input image
        out = gr.Image(label='Output') # show the fixed image
    # The second row will contain the buttons for the interface, along with their functionalities.
    with gr.Row():
        btn = gr.Button('Fix!') # define a button with text "Fix!"
        btn.click(fn=fixImg, inputs=inp, outputs=out) # upon clicking, take the input image and return the fixed image as the output
    
demo.launch()

* Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.
